# Module 4: Session Managers (10 min)

Add file-based persistence to the customer service agent. Stop the agent, restart it, and watch it remember the previous conversation.

**Prerequisites:** Modules 1-3 completed

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# AWS-sponsored events / AWS credits
# If you are running this workshop with AWS-provided credits, those credits
# only work with Amazon Nova models — not Claude (the default).
# To switch, import BedrockModel and pass it to Agent(...):
#
# from strands.models import BedrockModel
# model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# agent = Agent(model=model, tools=[...], session_manager=..., system_prompt=...)
#
# Available Nova model IDs: https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-amazon.html
#   amazon.nova-micro-v1:0  — fastest, text-only, lowest cost
#   amazon.nova-lite-v1:0   — low-cost, multimodal (text, image, video)
#   amazon.nova-pro-v1:0    — balanced accuracy/speed, multimodal (recommended)
#
# Run locally without AWS credentials using Ollama (https://ollama.com/download):
#   1. Install Ollama — macOS: DMG at ollama.com | Linux: curl -fsSL https://ollama.com/install.sh | sh
#   2. ollama pull llama3.1   (supports tool use)
#   3. pip install strands-agents[ollama]
#   4. from strands.models import OllamaModel
#      model = OllamaModel(host="http://localhost:11434", model_id="llama3.1")
#      agent = Agent(model=model, tools=[...], session_manager=..., system_prompt=...)
#   Other models with tool support: llama3.2, qwen2.5, qwen3, mistral

---

## Part 1: Agent Without Persistence (The Problem)

By default, agents lose their memory when you recreate them.

In [ ]:
from strands import Agent, AgentSkills
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

If there are previous messages in the conversation history, use that context
to continue helping the customer without asking them to repeat information."""

# First interaction
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)
agent("Hi, I'm customer C-1001. Can you look up my account?")

print(f"\n📝 Messages stored: {len(agent.messages)}")

In [ ]:
# Simulate a "restart" — create a new agent instance
agent2 = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

print(f"Messages after restart: {len(agent2.messages)}")
# The agent has no memory of the previous conversation!
agent2("What was my account status again?")

---

## Part 2: Add FileSessionManager

The `FileSessionManager` saves conversation history to disk. On restart, it reloads the messages.

The agent below uses **two** complementary pieces:
- **Session manager** (`FileSessionManager`) - persists the conversation *outside* the process so it survives restarts. This module's focus.
- **Conversation manager** (`SlidingWindowConversationManager`) - bounds what's kept *in context* on each call (here, the most recent 20 messages) so it doesn't grow without limit.

They solve different problems and work together: one stores history, the other trims what the model sees.

> **Tip:** The SDK also supports `context_manager="auto"`, which uses `SummarizingConversationManager` with proactive compression — a smarter alternative to sliding window. It's not yet in the official docs, but worth watching in the [Strands changelog](https://strandsagents.com/docs/changelog/).

In [ ]:
from strands.session.file_session_manager import FileSessionManager
from strands.agent.conversation_manager import SlidingWindowConversationManager

# Clean up any previous session files
import shutil, os
if os.path.exists("./sessions"):
    shutil.rmtree("./sessions")

session_manager = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    # Alternative: context_manager="auto" uses SummarizingConversationManager with
    # proactive compression — smarter than sliding window but not yet in the official docs.
    session_manager=session_manager,
)

# First interaction — this gets saved to disk
agent("Hi, I'm customer C-1001. Can you look up my account?")
print(f"\n📁 Session saved. Messages: {len(agent.messages)}")

---

## Part 3: Restart and Remember

Create a brand new agent with the same session ID. It should remember everything.

In [ ]:
# Simulate restart — new agent, same session_id
session_manager_2 = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent_restarted = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    # Alternative: context_manager="auto" uses SummarizingConversationManager with
    # proactive compression — smarter than sliding window but not yet in the official docs.
    session_manager=session_manager_2,
)

print(f"🔄 Restored messages: {len(agent_restarted.messages)}")
print("The agent remembers the previous conversation!\n")

# Ask a follow-up — the agent should know we're C-1001
agent_restarted("What orders do I have? You should already know my customer ID.")

---

## 🎯 Try It Yourself

Check what's stored on disk:

In [ ]:
import json

# FileSessionManager stores data in nested folders:
#   ./sessions/session_<id>/session.json
#   ./sessions/session_<id>/agents/agent_<id>/agent.json
#   ./sessions/session_<id>/agents/agent_<id>/messages/message_*.json
# So walk the tree recursively instead of listing only the top level.
session_dir = "./sessions"
for root, _dirs, files in os.walk(session_dir):
    for f in sorted(files):
        if f.endswith(".json"):
            filepath = os.path.join(root, f)
            rel = os.path.relpath(filepath, session_dir)
            print(f"📄 {rel} ({os.path.getsize(filepath)} bytes)")

# Count how many message files were persisted
message_files = [
    os.path.join(root, f)
    for root, _dirs, files in os.walk(session_dir)
    for f in files
    if f.startswith("message_") and f.endswith(".json")
]
print(f"\n💬 Messages persisted to disk: {len(message_files)}")

---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the **persistent** agent, run the companion script in a **terminal**. From the cloned repo:

```bash
cd samples/04-session-managers
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. Because it uses `FileSessionManager`, this is also the persistence demo: quit and run `python chat.py` again — it restores the earlier conversation. Use `--session-id <name>` to keep separate sessions.

---

## What's Next

The agent is persistent and follows rules — it's complete enough to ship. In **Module 5: Deploy**, you'll package this same agent and deploy it to Amazon Bedrock AgentCore Runtime with a single CLI command.